<a href="https://colab.research.google.com/github/fboliveira/Evolutionary-Computation/blob/main/Codes/TestFunction/ga_integer_tsp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Bibliotecas utilizadas
import random

In [ ]:
# ------------------------------------------------------
# GA - Representação inteira - TSP: lista sem repetição
# ------------------------------------------------------

# Número de variáveis: número de clientes a serem visitados
variaveis = 10

# Valores mínimo e máximo
min = 1
max = 10

# Tamanho do cromossomo - número de variáveis/clientes
tamanho_cromossomo = variaveis

# Ranking linear:
# Definir a pressão seletiva -> SP[1.0, 2.0]
SP = 1.2

# Parâmetros do GA:

# Quantos indivíduos serão utilziados - tamanho da populacao
tamanho_populacao = 10

# Critério de parada número máximo de gerações
geracoes = 10

# Taxa de crossover - operador principal do GA
taxa_crossover = 0.8

# Taxa de mutação - operador secundário do GA
taxa_mutacao = 0.05

# Mecanismos de busca local - ga com busca local: algoritmo memético / busca local = operador de educação
taxa_busca_local = 0.4
## Parâmetros específicos com a busca local
## VND - número de vizinhanças utilizadas - 2

In [ ]:
# Cálculo da Função Objetivo - TSP: matriz de distâncias - n * n
# - Representação inteira - o cálculo é direto: sem necessidade de transformações
def calcular_funcao_objetivo(individuo):

  # Com os valores reais, calcular o custo da função objetivo
  # sum(x_i^2)
  custo = 0

  for i, valor in enumerate(individuo):
    custo += (i + valor) ** 2

  return custo

In [ ]:
# Geração da população inicial - clássico: aleatória

# Lista (vetor/array) para representar o indivíduo: individuo = []
# População - conjunto de listas (listas de listas)
populacao = []

# Para cada indívuduo da população
# Gerar uma lista de 1 até n (clientes)
# Embaralha a lista
# range: [start, stop[ : 1-n => 1 até (n - 1)
for i in range(tamanho_populacao):
  individuo = []
  # Para cada valor do indivíduo - tamanho do cromossomo

  # Lista de 1 até n
  individuo = list( range(min, max + 1) )
  # for i in range(min, max + 1):
  #   individuo.append(i)

  # Embaralha a lista
  random.shuffle(individuo)

  # Calcular o custo para cada indivíduo
  custo = calcular_funcao_objetivo(individuo)

  # Adicionar o indivíduo e o respectivo custo na população inicial
  populacao.append((individuo, custo))

In [ ]:
# Para uma construção gulosa dos indivíduos
def construcao_gulosa():

  individuo = []

  # Primeiro cliente a ser visitado - aleatória
  primeiro = random.randint(min, max)

  individuo.append(primeiro)

  # Demais visitas - clientes
  # Selecionar o cliente mais próximo a partir da matriz de distâncias

  return individuo


In [ ]:
print(populacao)

[([3, 10, 5, 4, 9, 2, 8, 6, 1, 7], 1148), ([3, 6, 9, 1, 4, 5, 2, 8, 7, 10], 1234), ([4, 8, 10, 9, 6, 5, 3, 2, 7, 1], 1072), ([6, 9, 2, 8, 5, 3, 4, 1, 10, 7], 1162), ([4, 9, 10, 5, 6, 1, 8, 3, 7, 2], 1102), ([5, 3, 8, 10, 4, 2, 9, 7, 1, 6], 1150), ([3, 5, 8, 9, 1, 10, 4, 2, 6, 7], 1172), ([4, 8, 5, 7, 9, 1, 6, 3, 2, 10], 1156), ([4, 3, 9, 6, 10, 8, 2, 7, 1, 5], 1136), ([2, 5, 10, 9, 4, 8, 6, 7, 3, 1], 1122)]


In [ ]:
# Avaliação da população - ranking linear
# Ordenar a população do pior para o melhor

def avaliar_populacao(populacao):

  # Classificar a população (ordem decrescente): tupla[1] -> custo
  populacao.sort(key = lambda tupla : tupla[1], reverse=True)
  # print(populacao)

  fitness = []

  # fitness(i) = 2 - SP + 2 * (SP - 1) * ( ( i - 1 ) / ( nInd - 1 ) )
  for i in range(1, tamanho_populacao + 1):
    fitness_individuo = 2 - SP + 2 * ( SP - 1 ) * ( (i - 1) / ( tamanho_populacao - 1 ) )
    fitness.append(fitness_individuo)

  return fitness

In [ ]:
# Estrutura de repetição

# Custo da melhor solução - geração inicial: último indivíduo -> tupla[1]
fitness = avaliar_populacao(populacao)

melhor_custo_ate_agora = populacao[ tamanho_populacao - 1 ][1]

print('Custo da solução inicial: ', melhor_custo_ate_agora)

# Iterar sobre essas gerações - processo de evolução/melhoria da solução
for geracao in range(geracoes):

  # Melhor custo até agora
  melhor_custo_ate_agora = populacao[ tamanho_populacao - 1 ][1]
  print(f'Geração: {geracao} - custo: {melhor_custo_ate_agora}')

  # Seleção dos progenitores:
  # - Aleatória ?
  # - Seleção por torneio? Dois a dois

  # - Seleção por roleta - fitness
  proporcao_fitness = []
  # Exemplo slide - roleta
  # [ 14.4; 49.2;  5.5;  30.9 ]
  # [ 14.4; 63.6; 69.1; 100.0 ]
  soma_fitness = sum(fitness)
  soma_proporcao = 0.0
  for valor in fitness:
    # soma_proporcao = soma_proporcao + (valor / soma_fitness)
    soma_proporcao += (valor / soma_fitness)
    proporcao_fitness.append(soma_proporcao)

  # Descendentes
  descendentes = []

  # Fator de reprodução
  # Taxa de descendentes - taxa de crossover
  numero_descendentes = round( taxa_crossover * tamanho_populacao )

  # Processo de reprodução:
  for d in range(numero_descendentes):

    # Selecionar os progenitores a partir do fitness - critério método da roleta

    # - Selecionar o índice do primeiro progenitor:
    rnd_primeiro_progenitor = random.random() # 50%

    indice_primeiro_progenitor = 0

    for indice, valor in enumerate(proporcao_fitness):
      if ( rnd_primeiro_progenitor <= valor ):
        indice_primeiro_progenitor = indice # 1
        break

    # - Selecionar o índice do segundo progenitor:
    # -- Garantir índices diferentes para os progenitores
    indice_segundo_progenitor = indice_primeiro_progenitor

    while ( indice_segundo_progenitor == indice_primeiro_progenitor ):

      rnd_segundo_progenitor = random.random() # 80%

      for indice, valor in enumerate(proporcao_fitness):
        if ( rnd_segundo_progenitor <= valor ):
          indice_segundo_progenitor = indice # 3
          break

    # - Recuperar da população os progenitores - tupla[0]
    primeiro_progenitor = populacao[ indice_primeiro_progenitor ][0]
    segundo_progenitor = populacao[ indice_segundo_progenitor ][0]

    # - Operação de crossover: Operador OX
    # Dois pontos de corte, considerando que ponto1 < ponto2
    # Intevalo: [2, n - 2]
    pontos_de_corte = random.sample( range(2, tamanho_cromossomo - 2) , 2)
    # Ordenar os pontos para garantir que ponto1 < ponto2
    pontos_de_corte.sort()
    ponto1 = pontos_de_corte[0]
    ponto2 = pontos_de_corte[1]

    # -- Primeiro descendente
    # Gerando os descendentes conforme o tamanho do cromosso com valor iguais a zero
    # D  = [ 0 0 0 0 0 0 0 0 0 ]
    primeiro_descendente = [0] * tamanho_cromossomo
    segundo_descendente = [0] * tamanho_cromossomo

    # D1 = [ _ _ _ 2 4 1 _ _ _ ]
    # D2 = [ _ _ _ 4 6 5 _ _ _ ]
    # Copiar para o D1 a sequência entre ponto1 e ponto2 do P1
    # Copiar para o D2 a sequência entre ponto1 e ponto2 do P2
    for i in range(ponto1, ponto2):
      primeiro_descendente[i] = primeiro_progenitor[i]
      segundo_descendente[i] = segundo_progenitor[i]

    # Para gerar/atualizar o D1:
    # Ordem_P2 = [8 9 3] + [1 2 7 4 6 5]
    # Atualizando em D1 conforme a ordem, considerando os que ainda não existem.
    ordem_visitacao_p2 = segundo_progenitor[ponto2:tamanho_cromossomo] + segundo_progenitor[0:ponto2]

    posicao = ponto2

    for cliente in ordem_visitacao_p2:
      # Se o cliente não existe no descendente 1, inseri-lo na posicao
      if cliente not in primeiro_descendente:
        primeiro_descendente[posicao] = cliente
        posicao = posicao + 1

        if posicao == tamanho_cromossomo:
          posicao = 0

    # Para gerar/atualizar o D2:
    # Ordem_P1 = [5 7 9] + [6 3 8 2 4 1]
    ordem_visitacao_p1 = primeiro_progenitor[ponto2:tamanho_cromossomo] + primeiro_progenitor[0:ponto2]

    # Inicia a atualização após o segundo corte (a sequência entre os cortes já foi inserida)
    posicao = ponto2
    for cliente in ordem_visitacao_p1:
      # Se não existe em D2, atualiza na posicao
      if cliente not in segundo_descendente:
        segundo_descendente[posicao] = cliente
        posicao += 1

        if posicao == tamanho_cromossomo:
          posicao = 0

    # Mutação - operador secundário
    # - Ocorre nos descendentes após (em seguida) ao processo de crossover

    # -- Possibilidade de mutação de apenas UMA troca
    # -- Critério de mutação troca(i, j): swap
    probabilidade_mutacao = random.random()

    if (probabilidade_mutacao <= taxa_mutacao):

      # Aplicar apenas uma troca em duas posições aleatórias: i, j
      i = random.randint(0, len(primeiro_descendente) - 1) # [a, b]

      j = i
      # Garantir que i != j
      while (i == j):
        j = random.randint(0, len(primeiro_descendente) - 1) # [a, b]

      # Trocar os valores (clientes) entre as posições i e j
      aux = primeiro_descendente[i]
      primeiro_descendente[i] = primeiro_descendente[j]
      primeiro_descendente[j] = aux

    # -- Possibilidade de mutação por posição (i) - para cada uma das posições
    for posicao in range(len(segundo_descendente)):
      probabilidade_mutacao = random.random()
      i = posicao

      if ( probabilidade_mutacao <= taxa_mutacao ):

        # Selecionar a posição j diferente de i
        j = i

        # Garantir que j != i
        while (j == i):
          j = random.randint(0, len(primeiro_descendente) - 1) # [a, b]

        # Trocar os valores (clientes) entre as posições i e j
        segundo_progenitor[i], segundo_progenitor[j] = segundo_progenitor[j], segundo_progenitor[i]

    # -- Encerra o processo de reprodução para os dois desecendentes

    # Calcular o custo dos descendentes
    custo_primeiro_descendente = calcular_funcao_objetivo(primeiro_descendente)
    custo_segundo_descendente = calcular_funcao_objetivo(segundo_descendente)

    # GA Memético -> GA com busca local
    # -- processo de busca de local, considerando uma taxa de probabalilidade para cada um deles.
    # -- TSP/VRP, problema combinatório: representação inteira.
    if random.random() <= taxa_busca_local:
      # Busca local em D1
      # busca_local(primeiro_descendente)
      pass

    if random.random() <= taxa_busca_local:
      # Busca local em D1
      # busca_local(segundo_descendente)
      # Delta -> swap(i, j)
      # individuo = [ ..., i, ..., j, ... ]
      # dist = dist - d(i - 1, i) - d(i, i + i) + ...
      # Calcular fo -> O(n)
      # Calcular delta: O(1)
      pass

    # Adicionar os descendentes na lista -> definir os sobreviventes
    descendentes.append( (primeiro_descendente, custo_primeiro_descendente) )
    descendentes.append( (segundo_descendente, custo_segundo_descendente) )

    # -- Encerra o for do processo de reprodução para população

  # Avaliação da população: a população atual e os descedentes
  # -- Combinar as duas populações: atual + descedentes (merge)
  nova_populacao = populacao + descendentes

  # -- Ordenação da nova populacao [do melhor para o pior]
  nova_populacao.sort(key = lambda tupla : tupla[1]) # asc [melhor para o pior]

  # Definir os sobreviventes:
  # -- Fitness e seleção por roleta (bons e ruins) -> não se tem garantir que o melhor indivíduo será selecionado
  # -- Parte elitismo (5%), parte fitness/roleta (75%), parte torneio (20%)
  # -- Substituir parte da população por novos indivíduos (10% aleatória) -> garantir diversidade na população
  # -- Selecionar os n primeiros melhores (tamanho_populacao) -> 100% elitismo
  populacao = nova_populacao[0:tamanho_populacao]

  # Avaliar a população sobrevivente
  fitness = avaliar_populacao(populacao) # [do pior para o melhor]


Custo da solução inicial:  1072
Geração: 0 - custo: 1072
Geração: 1 - custo: 1046
Geração: 2 - custo: 1024
Geração: 3 - custo: 1024
Geração: 4 - custo: 1024
Geração: 5 - custo: 1020
Geração: 6 - custo: 1020
Geração: 7 - custo: 1016
Geração: 8 - custo: 1016
Geração: 9 - custo: 1016


In [ ]:
# Lista o melhor indivíduo
print(populacao)

# Melhor resultado:
melhor_individuo = populacao[ tamanho_populacao - 1 ][0]
melhor_custo = populacao[ tamanho_populacao - 1 ][1]

print("Melhor custo: ", melhor_custo)
print("Melhor individuo: ", melhor_individuo)

[([8, 10, 3, 5, 4, 9, 6, 7, 1, 2], 1026), ([10, 9, 7, 5, 4, 3, 6, 1, 8, 2], 1024), ([6, 7, 9, 10, 8, 4, 1, 5, 2, 3], 1020), ([10, 8, 9, 5, 2, 4, 6, 3, 1, 7], 1020), ([9, 10, 7, 5, 8, 4, 6, 3, 1, 2], 1018), ([10, 9, 7, 5, 8, 4, 6, 3, 1, 2], 1016), ([10, 9, 7, 5, 8, 4, 6, 3, 1, 2], 1016), ([10, 9, 7, 5, 8, 4, 6, 3, 1, 2], 1016), ([10, 9, 7, 5, 8, 4, 6, 3, 1, 2], 1016), ([10, 9, 7, 5, 8, 4, 6, 3, 1, 2], 1016)]
Melhor custo:  1016
Melhor individuo:  [10, 9, 7, 5, 8, 4, 6, 3, 1, 2]
